In [9]:
from __future__ import annotations

from collections import defaultdict, deque
from pathlib import Path
from typing import Dict, Mapping, Set

import numpy as np
import pandas as pd

RANDOM_SEED = 42
N_EMPLOYEES = 80
N_PROJECTS = 120
SIM_WEEKS = 20

DEPARTMENTS = ["Hardware", "Software", "Testing", "Integration"]

ISSUE_LAMBDA = 0.35
REWORK_HOURS_RANGE = (4.0, 18.0)
WEEKLY_OVERDRIVE = 1.10

# Dependency softening controls for more organic portfolio behavior
DEPENDENT_PROJECT_RATIO = 0.20
DEPENDENCY_PROGRESS_CAP = 40.0
DEPENDENCY_REQUIRED_PCT = 100.0

# Forecast controls
RECENT_BURN_WINDOW = 4
MIN_BURN_FLOOR_RATIO = 0.10

EPS = 1e-9

In [10]:
class Project:
    def __init__(
        self,
        project_id: int,
        department: str,
        total_required_hours: float,
        planned_duration_weeks: int,
    ) -> None:
        if total_required_hours <= 0:
            raise ValueError("total_required_hours must be > 0")
        if planned_duration_weeks <= 0:
            raise ValueError("planned_duration_weeks must be > 0")

        self.project_id = int(project_id)
        self.department = str(department)
        self.initial_required_hours = float(total_required_hours)
        self.total_required_hours = float(total_required_hours)
        self.planned_duration_weeks = int(planned_duration_weeks)

        # Freeze project weekly throughput baseline; rework does not increase this.
        self.frozen_weekly_capacity = self.initial_required_hours / max(1, self.planned_duration_weeks)

        self.completed_hours = 0.0
        self.dependencies: Dict[int, Dict[str, float]] = {}
        self._weekly_completed_hours: Dict[int, float] = defaultdict(float)

    def reset_progress(self) -> None:
        self.total_required_hours = float(self.initial_required_hours)
        self.completed_hours = 0.0
        self._weekly_completed_hours.clear()

    @property
    def progress_pct(self) -> float:
        if self.total_required_hours <= EPS:
            return 100.0
        return min(100.0, (self.completed_hours / self.total_required_hours) * 100.0)

    @property
    def remaining_hours(self) -> float:
        return max(0.0, self.total_required_hours - self.completed_hours)

    @property
    def planned_weekly_hours(self) -> float:
        return self.frozen_weekly_capacity

    def add_dependency(
        self,
        predecessor_project_id: int,
        self_progress_cap_pct: float = 40.0,
        predecessor_required_pct: float = 100.0,
    ) -> None:
        predecessor_project_id = int(predecessor_project_id)
        if predecessor_project_id == self.project_id:
            raise ValueError("A project cannot depend on itself")
        if not (0.0 <= self_progress_cap_pct <= 100.0):
            raise ValueError("self_progress_cap_pct must be in [0, 100]")
        if not (0.0 <= predecessor_required_pct <= 100.0):
            raise ValueError("predecessor_required_pct must be in [0, 100]")

        self.dependencies[predecessor_project_id] = {
            "self_progress_cap_pct": float(self_progress_cap_pct),
            "predecessor_required_pct": float(predecessor_required_pct),
        }

    def _allowed_progress_cap_pct(self, projects_by_id: Mapping[int, "Project"]) -> float:
        cap_pct = 100.0
        for predecessor_id, rule in self.dependencies.items():
            predecessor = projects_by_id.get(predecessor_id)
            predecessor_progress = predecessor.progress_pct if predecessor is not None else 0.0
            if predecessor_progress + EPS < rule["predecessor_required_pct"]:
                cap_pct = min(cap_pct, rule["self_progress_cap_pct"])
        return cap_pct

    def remaining_hours_until_dependency_cap(self, projects_by_id: Mapping[int, "Project"]) -> float:
        allowed_progress_pct = self._allowed_progress_cap_pct(projects_by_id)
        allowed_hours = (allowed_progress_pct / 100.0) * self.total_required_hours
        return max(0.0, allowed_hours - self.completed_hours)

    def remaining_weekly_throughput(self, week: int, weekly_overdrive: float = 1.10) -> float:
        weekly_cap = self.planned_weekly_hours * max(0.0, weekly_overdrive)
        used_this_week = self._weekly_completed_hours[week]
        return max(0.0, weekly_cap - used_this_week)

    def recent_burn_rate(self, week: int, burn_window: int = 4) -> float:
        start = max(1, week - burn_window + 1)
        observed = [self._weekly_completed_hours[w] for w in range(start, week + 1)]
        return float(sum(observed) / len(observed)) if observed else 0.0

    def allocatable_hours(
        self,
        requested_hours: float,
        projects_by_id: Mapping[int, "Project"],
        week: int,
        weekly_overdrive: float = 1.10,
    ) -> float:
        if requested_hours <= 0:
            return 0.0

        return max(
            0.0,
            min(
                float(requested_hours),
                self.remaining_hours,
                self.remaining_hours_until_dependency_cap(projects_by_id),
                self.remaining_weekly_throughput(week=week, weekly_overdrive=weekly_overdrive),
            ),
        )

    def apply_work(
        self,
        requested_hours: float,
        projects_by_id: Mapping[int, "Project"],
        week: int,
        weekly_overdrive: float = 1.10,
    ) -> float:
        allocated = self.allocatable_hours(
            requested_hours=requested_hours,
            projects_by_id=projects_by_id,
            week=week,
            weekly_overdrive=weekly_overdrive,
        )
        if allocated <= 0:
            return 0.0

        self.completed_hours += allocated
        self._weekly_completed_hours[week] += allocated
        return allocated


class Employee:
    CONTEXT_SWITCH_PENALTY = 0.15

    def __init__(self, employee_id: int, department: str, base_capacity_hours: float) -> None:
        if base_capacity_hours <= 0:
            raise ValueError("base_capacity_hours must be > 0")

        self.employee_id = int(employee_id)
        self.department = str(department)
        self.base_capacity_hours = float(base_capacity_hours)

        self._weekly_projects: Dict[int, Set[int]] = defaultdict(set)
        self._weekly_allocated_hours: Dict[int, float] = defaultdict(float)

    def reset_tracking(self) -> None:
        self._weekly_projects.clear()
        self._weekly_allocated_hours.clear()

    def _project_count_with_candidate(self, week: int, candidate_project_id: int | None = None) -> int:
        projects = set(self._weekly_projects.get(week, set()))
        if candidate_project_id is not None:
            projects.add(int(candidate_project_id))
        return len(projects)

    def effective_capacity(self, week: int, candidate_project_id: int | None = None) -> float:
        n_projects = self._project_count_with_candidate(week=week, candidate_project_id=candidate_project_id)
        penalty = self.CONTEXT_SWITCH_PENALTY if n_projects > 2 else 0.0
        return self.base_capacity_hours * (1.0 - penalty)

    def remaining_capacity(self, week: int, candidate_project_id: int | None = None) -> float:
        cap = self.effective_capacity(week=week, candidate_project_id=candidate_project_id)
        used = self._weekly_allocated_hours.get(week, 0.0)
        return max(0.0, cap - used)

    def register_allocation(self, week: int, project_id: int, requested_hours: float) -> float:
        if requested_hours <= 0:
            return 0.0

        available = self.remaining_capacity(week=week, candidate_project_id=project_id)
        allocated = min(float(requested_hours), available)
        if allocated <= 0:
            return 0.0

        self._weekly_projects[week].add(int(project_id))
        self._weekly_allocated_hours[week] += allocated

        cap_after = self.effective_capacity(week=week)
        if self._weekly_allocated_hours[week] > cap_after + EPS:
            raise RuntimeError("Employee weekly capacity exceeded. Allocation logic is inconsistent.")

        return allocated

    def week_state(self, week: int) -> dict:
        used = float(self._weekly_allocated_hours.get(week, 0.0))
        n_projects = len(self._weekly_projects.get(week, set()))
        cap = float(self.effective_capacity(week=week))
        utilization_pct = 0.0 if cap <= EPS else min(100.0, (used / cap) * 100.0)

        return {
            "employee_id": self.employee_id,
            "department": self.department,
            "week": week,
            "projects_assigned": n_projects,
            "used_hours": round(used, 4),
            "effective_capacity": round(cap, 4),
            "remaining_capacity": round(max(0.0, cap - used), 4),
            "utilization_pct": round(utilization_pct, 2),
            "capacity_exceeded": bool(used > cap + EPS),
        }

In [11]:
class SimulationEngine:
    def __init__(
        self,
        projects: list[Project],
        employees: list[Employee],
        issue_lambda: float = ISSUE_LAMBDA,
        rework_hours_range: tuple[float, float] = REWORK_HOURS_RANGE,
        weekly_overdrive: float = WEEKLY_OVERDRIVE,
        burn_window: int = RECENT_BURN_WINDOW,
        min_burn_floor_ratio: float = MIN_BURN_FLOOR_RATIO,
        rng: np.random.Generator | None = None,
    ) -> None:
        self.projects = list(projects)
        self.employees = list(employees)
        self.projects_by_id = {p.project_id: p for p in self.projects}
        self.issue_lambda = float(issue_lambda)
        self.rework_hours_range = tuple(rework_hours_range)
        self.weekly_overdrive = float(weekly_overdrive)
        self.burn_window = int(max(1, burn_window))
        self.min_burn_floor_ratio = float(max(0.0, min_burn_floor_ratio))
        self.rng = rng if rng is not None else np.random.default_rng()
        self.history_log: list[dict] = []

        if len(self.projects_by_id) != len(self.projects):
            raise ValueError("Project IDs must be unique")

        self.successors = self._build_successor_graph()
        self._validate_dag()

    def _build_successor_graph(self) -> dict[int, set[int]]:
        successors: dict[int, set[int]] = {pid: set() for pid in self.projects_by_id}
        for project in self.projects:
            for predecessor_id in project.dependencies:
                if predecessor_id not in self.projects_by_id:
                    raise ValueError(f"Project {project.project_id} has unknown predecessor {predecessor_id}")
                successors[predecessor_id].add(project.project_id)
        return successors

    def _validate_dag(self) -> None:
        indegree = {pid: 0 for pid in self.projects_by_id}
        for project in self.projects:
            for predecessor_id in project.dependencies:
                indegree[project.project_id] += 1

        queue = deque([pid for pid, deg in indegree.items() if deg == 0])
        visited = 0
        while queue:
            node = queue.popleft()
            visited += 1
            for child in self.successors.get(node, set()):
                indegree[child] -= 1
                if indegree[child] == 0:
                    queue.append(child)

        if visited != len(self.projects):
            raise ValueError("Dependency graph contains a cycle. DAG is required.")

    def _reset_run_state(self) -> None:
        self.history_log = []
        for p in self.projects:
            p.reset_progress()
        for e in self.employees:
            e.reset_tracking()

    def _descendant_count(self, project_id: int, memo: dict[int, int]) -> int:
        if project_id in memo:
            return memo[project_id]
        total = 0
        for child_id in self.successors.get(project_id, set()):
            total += 1 + self._descendant_count(child_id, memo)
        memo[project_id] = total
        return total

    def _blocked_successor_count(self, project_id: int) -> int:
        blocked = 0
        parent = self.projects_by_id[project_id]
        for child_id in self.successors.get(project_id, set()):
            child = self.projects_by_id[child_id]
            if child.remaining_hours <= EPS:
                continue

            rule = child.dependencies.get(project_id)
            if rule is None:
                continue

            parent_not_ready = parent.progress_pct + EPS < rule["predecessor_required_pct"]
            child_hard_capped = child.remaining_hours_until_dependency_cap(self.projects_by_id) <= EPS
            if parent_not_ready and child_hard_capped:
                blocked += 1

        return blocked

    def _critical_path_score(self, project: Project, memo: dict[int, int]) -> tuple[float, ...]:
        blocked_successors = self._blocked_successor_count(project.project_id)
        descendants = self._descendant_count(project.project_id, memo)
        return (
            float(blocked_successors),
            float(descendants),
            float(project.remaining_hours),
            float(100.0 - project.progress_pct),
        )

    def _rank_projects_for_employee(self, employee: Employee, week: int) -> list[Project]:
        memo: dict[int, int] = {}
        ranked: list[tuple[tuple[float, ...], Project]] = []

        for project in self.projects:
            if project.department != employee.department:
                continue
            if project.remaining_hours <= EPS:
                continue

            alloc_now = project.allocatable_hours(
                requested_hours=10**9,
                projects_by_id=self.projects_by_id,
                week=week,
                weekly_overdrive=self.weekly_overdrive,
            )
            if alloc_now <= EPS:
                continue

            score = self._critical_path_score(project, memo)
            ranked.append((score, project))

        ranked.sort(key=lambda x: x[0], reverse=True)
        return [item[1] for item in ranked]

    def _apply_operational_issues(self) -> dict[int, dict[str, float]]:
        low, high = self.rework_hours_range
        issue_map: dict[int, dict[str, float]] = {}

        for project in self.projects:
            if project.remaining_hours <= EPS:
                issue_map[project.project_id] = {"issues": 0, "extra_rework_hours": 0.0}
                continue

            n_issues = int(self.rng.poisson(self.issue_lambda))
            extra_rework = 0.0
            if n_issues > 0:
                extra_rework = float(self.rng.uniform(low, high, size=n_issues).sum())
                project.total_required_hours += extra_rework

            issue_map[project.project_id] = {
                "issues": n_issues,
                "extra_rework_hours": round(extra_rework, 2),
            }

        return issue_map

    def _allocate_week(self, week: int) -> None:
        employee_order = list(self.employees)
        self.rng.shuffle(employee_order)

        for employee in employee_order:
            while employee.remaining_capacity(week) > EPS:
                candidates = self._rank_projects_for_employee(employee=employee, week=week)
                if not candidates:
                    break

                allocated_any = False
                for project in candidates:
                    emp_available = employee.remaining_capacity(week, candidate_project_id=project.project_id)
                    if emp_available <= EPS:
                        continue

                    proj_allocatable = project.allocatable_hours(
                        requested_hours=emp_available,
                        projects_by_id=self.projects_by_id,
                        week=week,
                        weekly_overdrive=self.weekly_overdrive,
                    )
                    if proj_allocatable <= EPS:
                        continue

                    emp_alloc = employee.register_allocation(
                        week=week,
                        project_id=project.project_id,
                        requested_hours=proj_allocatable,
                    )
                    if emp_alloc <= EPS:
                        continue

                    actual = project.apply_work(
                        requested_hours=emp_alloc,
                        projects_by_id=self.projects_by_id,
                        week=week,
                        weekly_overdrive=self.weekly_overdrive,
                    )
                    if actual <= EPS:
                        continue

                    allocated_any = True
                    break

                if not allocated_any:
                    break

    def _project_forecast_metrics(self, project: Project, week: int) -> dict:
        baseline_weekly = max(project.planned_weekly_hours, EPS)
        recent_burn = project.recent_burn_rate(week=week, burn_window=self.burn_window)
        burn_floor = baseline_weekly * self.min_burn_floor_ratio
        effective_burn = max(recent_burn, burn_floor, EPS)

        if project.remaining_hours <= EPS:
            forecast_finish_week = float(week)
        else:
            forecast_finish_week = float(week + (project.remaining_hours / effective_burn))

        forecast_delay_weeks = max(0.0, forecast_finish_week - float(project.planned_duration_weeks))
        forecast_delay_pct = min(100.0, (forecast_delay_weeks / max(1, project.planned_duration_weeks)) * 100.0)

        return {
            "recent_burn_rate": round(recent_burn, 4),
            "forecast_burn_rate": round(effective_burn, 4),
            "forecast_finish_week": round(forecast_finish_week, 2),
            "forecast_delay_weeks": round(forecast_delay_weeks, 2),
            "delay_pct": round(forecast_delay_pct, 2),
        }

    def _week_state(self, week: int, issue_map: dict[int, dict[str, float]]) -> dict:
        project_states = []
        active_projects = []

        for p in self.projects:
            forecast = self._project_forecast_metrics(project=p, week=week)

            if p.remaining_hours > EPS:
                active_projects.append(p.project_id)

            project_states.append(
                {
                    "project_id": p.project_id,
                    "department": p.department,
                    "completed_hours": round(p.completed_hours, 2),
                    "remaining_hours": round(p.remaining_hours, 2),
                    "total_required_hours": round(p.total_required_hours, 2),
                    "progress_pct": round(p.progress_pct, 2),
                    "delay_pct": forecast["delay_pct"],
                    "forecast_finish_week": forecast["forecast_finish_week"],
                    "forecast_delay_weeks": forecast["forecast_delay_weeks"],
                    "recent_burn_rate": forecast["recent_burn_rate"],
                    "forecast_burn_rate": forecast["forecast_burn_rate"],
                    "is_active": p.remaining_hours > EPS,
                }
            )

        employee_week_states = [e.week_state(week=week) for e in self.employees]

        critical_path_projects = [
            p.project_id
            for p in self.projects
            if p.remaining_hours > EPS and self._blocked_successor_count(p.project_id) > 0
        ]

        portfolio_completed = float(sum(p.completed_hours for p in self.projects))
        portfolio_required = float(sum(p.total_required_hours for p in self.projects))

        return {
            "week": week,
            "active_projects": active_projects,
            "critical_path_projects": critical_path_projects,
            "portfolio_completed_hours": round(portfolio_completed, 2),
            "portfolio_total_required_hours": round(portfolio_required, 2),
            "operational_issues": issue_map,
            "project_states": project_states,
            "employee_week_states": employee_week_states,
        }

    def run_simulation(self, weeks: int) -> list[dict]:
        if weeks <= 0:
            raise ValueError("weeks must be > 0")

        self._reset_run_state()

        for week in range(1, weeks + 1):
            issue_map = self._apply_operational_issues()
            self._allocate_week(week)
            self.history_log.append(self._week_state(week, issue_map))

        return self.history_log

In [12]:
rng = np.random.default_rng(RANDOM_SEED)

# 1) Generate 80 employees
employee_departments = rng.choice(DEPARTMENTS, size=N_EMPLOYEES, replace=True)
base_capacities = np.clip(rng.normal(loc=38.0, scale=3.0, size=N_EMPLOYEES), 30.0, 42.0)

employees = [
    Employee(
        employee_id=i + 1,
        department=str(employee_departments[i]),
        base_capacity_hours=float(base_capacities[i]),
    )
    for i in range(N_EMPLOYEES)
]

# 2) Generate 120 projects with gamma durations
project_departments = rng.choice(DEPARTMENTS, size=N_PROJECTS, replace=True)
durations = rng.gamma(shape=4.0, scale=6.0, size=N_PROJECTS)
durations = np.clip(np.rint(durations), 12, 52).astype(int)

# Purposefully substantial work volume so capacity constraints are visible and realistic
base_weekly_load = np.clip(rng.normal(loc=72.0, scale=14.0, size=N_PROJECTS), 45.0, 110.0)
complexity_factor = rng.uniform(1.0, 1.5, size=N_PROJECTS)
total_required_hours = np.maximum(120.0, durations * base_weekly_load * complexity_factor)

projects = [
    Project(
        project_id=i + 1,
        department=str(project_departments[i]),
        total_required_hours=float(total_required_hours[i]),
        planned_duration_weeks=int(durations[i]),
    )
    for i in range(N_PROJECTS)
]
projects_by_id = {p.project_id: p for p in projects}

# 3) Random DAG dependencies with softened strictness
n_dependent_projects = int(DEPENDENT_PROJECT_RATIO * N_PROJECTS)
dependent_project_ids = rng.choice(
    np.arange(2, N_PROJECTS + 1),
    size=n_dependent_projects,
    replace=False,
)

for child_id in dependent_project_ids:
    child = projects_by_id[int(child_id)]
    predecessor_candidates = np.arange(1, int(child_id))
    n_preds = int(min(len(predecessor_candidates), rng.integers(1, 4)))
    predecessors = rng.choice(predecessor_candidates, size=n_preds, replace=False)

    for pred_id in np.atleast_1d(predecessors):
        child.add_dependency(
            predecessor_project_id=int(pred_id),
            self_progress_cap_pct=DEPENDENCY_PROGRESS_CAP,
            predecessor_required_pct=DEPENDENCY_REQUIRED_PCT,
        )

# 4) Run simulation
engine = SimulationEngine(
    projects=projects,
    employees=employees,
    issue_lambda=ISSUE_LAMBDA,
    rework_hours_range=REWORK_HOURS_RANGE,
    weekly_overdrive=WEEKLY_OVERDRIVE,
    burn_window=RECENT_BURN_WINDOW,
    min_burn_floor_ratio=MIN_BURN_FLOOR_RATIO,
    rng=rng,
)

history_log = engine.run_simulation(weeks=SIM_WEEKS)

# 5) Export final project states to CSV
final_project_states = pd.DataFrame(history_log[-1]["project_states"])

# 6) Export employee-week capacity audit to CSV
employee_capacity_rows = []
for week_state in history_log:
    for row in week_state["employee_week_states"]:
        employee_capacity_rows.append(row)

employee_capacity_audit = pd.DataFrame(employee_capacity_rows)

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

output_dir = workspace_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)

project_output_path = output_dir / "enterprise_portfolio_results.csv"
audit_output_path = output_dir / "employee_capacity_audit.csv"

final_project_states.to_csv(project_output_path, index=False)
employee_capacity_audit.to_csv(audit_output_path, index=False)

capacity_violations = int(employee_capacity_audit["capacity_exceeded"].sum())

print(f"Saved: {project_output_path}")
print(f"Saved: {audit_output_path}")
print(f"Project rows: {len(final_project_states)}")
print(f"Employee-week audit rows: {len(employee_capacity_audit)}")
print(f"Active projects after {SIM_WEEKS} weeks: {int(final_project_states['is_active'].sum())}")
print(f"Capacity violations detected: {capacity_violations}")

final_project_states.head()

Saved: /workspaces/codespaces-jupyter/data/enterprise_portfolio_results.csv
Saved: /workspaces/codespaces-jupyter/data/employee_capacity_audit.csv
Project rows: 120
Employee-week audit rows: 1600
Active projects after 20 weeks: 114
Capacity violations detected: 0


,project_id,department,completed_hours,remaining_hours,total_required_hours,progress_pct,delay_pct,forecast_finish_week,forecast_delay_weeks,recent_burn_rate,forecast_burn_rate,is_active
0,1,Testing,1855.34,320.36,2175.70,85.28,0.00,23.45,0.00,92.7670,92.7670,True
1,2,Hardware,1670.46,312.45,1982.91,84.24,0.00,23.74,0.00,83.5229,83.5229,True
2,3,Integration,1306.53,1959.79,3266.32,40.00,100.00,248.71,211.71,0.8594,8.5688,True
3,4,Software,2389.94,2586.68,4976.62,48.02,0.00,41.65,0.00,119.4970,119.4970,True
4,5,Integration,1456.26,0.00,1456.26,100.00,66.67,20.00,8.00,0.0000,11.7150,False


In [13]:
import io
import json
import sqlite3
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd


# -----------------------------
# Config
# -----------------------------
WORKSPACE = Path("/workspaces/codespaces-jupyter")
DATA_DIR = WORKSPACE / "data"
NOTEBOOK_PATH = WORKSPACE / "notebooks" / "simulasyon copy.ipynb"
DB_PATH = WORKSPACE / "portfolio.db"

PROJECTS_CSV = DATA_DIR / "enterprise_portfolio_results.csv"
CAPACITY_CSV = DATA_DIR / "employee_capacity_audit.csv"


def replay_notebook_for_dependency_and_rework(
    notebook_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Replays first 4 cells of the notebook in an isolated namespace to reconstruct:
    1) project_dependencies table
    2) weekly_rework table
    """
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace: dict = {}

    # Suppress notebook print output while replaying.
    with redirect_stdout(io.StringIO()):
        for cell in nb["cells"][:4]:
            if cell.get("cell_type") != "code":
                continue
            code = "\n".join(cell.get("source", []))
            exec(code, namespace)

    projects_runtime = namespace.get("projects", [])
    history_log = namespace.get("history_log", [])

    # Build dependency edges from runtime project objects.
    dep_rows = []
    for p in projects_runtime:
        for pred_id in p.dependencies.keys():
            dep_rows.append(
                {
                    "child_project_id": int(p.project_id),
                    "predecessor_project_id": int(pred_id),
                }
            )
    dependencies_df = pd.DataFrame(
        dep_rows, columns=["child_project_id", "predecessor_project_id"]
    )

    # Build weekly total rework from history_log operational_issues.
    rework_rows = []
    for wk in history_log:
        week_no = int(wk["week"])
        total_rework = sum(
            float(v.get("extra_rework_hours", 0.0))
            for v in wk.get("operational_issues", {}).values()
        )
        rework_rows.append(
            {
                "week": week_no,
                "total_rework_hours": round(total_rework, 4),
            }
        )
    weekly_rework_df = pd.DataFrame(rework_rows, columns=["week", "total_rework_hours"])

    return dependencies_df, weekly_rework_df


def main() -> None:
    # 1) Create/connect SQLite DB
    projects_df = pd.read_csv(PROJECTS_CSV)
    capacity_df = pd.read_csv(CAPACITY_CSV)

    dependencies_df, weekly_rework_df = replay_notebook_for_dependency_and_rework(NOTEBOOK_PATH)

    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")

        # 2) Load data into SQL tables
        projects_df.to_sql("projects", conn, if_exists="replace", index=False)
        capacity_df.to_sql("employee_capacity_audit", conn, if_exists="replace", index=False)
        dependencies_df.to_sql("project_dependencies", conn, if_exists="replace", index=False)
        weekly_rework_df.to_sql("weekly_rework", conn, if_exists="replace", index=False)

        # Helpful indexes
        conn.executescript(
            """
            CREATE INDEX IF NOT EXISTS idx_projects_project_id ON projects(project_id);
            CREATE INDEX IF NOT EXISTS idx_projects_department ON projects(department);
            CREATE INDEX IF NOT EXISTS idx_capacity_department ON employee_capacity_audit(department);
            CREATE INDEX IF NOT EXISTS idx_capacity_week ON employee_capacity_audit(week);
            CREATE INDEX IF NOT EXISTS idx_dep_child ON project_dependencies(child_project_id);
            CREATE INDEX IF NOT EXISTS idx_dep_pred ON project_dependencies(predecessor_project_id);
            CREATE INDEX IF NOT EXISTS idx_rework_week ON weekly_rework(week);
            """
        )

        # -----------------------------
        # Query A:
        # Which department has highest total Risk Exposure
        # and what is avg employee utilization?
        # -----------------------------
        query_a = """
        WITH project_risk AS (
            SELECT
                department,
                project_id,
                (total_required_hours * (delay_pct / 100.0)) AS risk_exposure
            FROM projects
        ),
        dept_risk AS (
            SELECT
                department,
                SUM(risk_exposure) AS total_risk_exposure
            FROM project_risk
            GROUP BY department
        ),
        dept_util AS (
            SELECT
                department,
                AVG(utilization_pct) AS avg_employee_utilization
            FROM employee_capacity_audit
            GROUP BY department
        ),
        ranked AS (
            SELECT
                dr.department,
                dr.total_risk_exposure,
                du.avg_employee_utilization,
                DENSE_RANK() OVER (ORDER BY dr.total_risk_exposure DESC) AS risk_rank
            FROM dept_risk dr
            LEFT JOIN dept_util du ON du.department = dr.department
        )
        SELECT
            department,
            ROUND(total_risk_exposure, 2) AS total_risk_exposure,
            ROUND(avg_employee_utilization, 2) AS avg_employee_utilization
        FROM ranked
        WHERE risk_rank = 1;
        """

        # -----------------------------
        # Query B:
        # Top 5 most "At Risk" projects by forecast delay
        # with predecessor status.
        # -----------------------------
        query_b = """
        WITH ranked_projects AS (
            SELECT
                p.project_id,
                p.department,
                p.progress_pct,
                p.delay_pct,
                p.forecast_delay_weeks,
                p.remaining_hours,
                DENSE_RANK() OVER (
                    ORDER BY p.forecast_delay_weeks DESC, p.delay_pct DESC, p.remaining_hours DESC
                ) AS risk_rank
            FROM projects p
        ),
        top5 AS (
            SELECT *
            FROM ranked_projects
            WHERE risk_rank <= 5
        ),
        predecessor_status AS (
            SELECT
                d.child_project_id,
                d.predecessor_project_id,
                pp.progress_pct AS predecessor_progress_pct,
                CASE
                    WHEN pp.progress_pct >= 100 THEN 'Cleared'
                    WHEN pp.progress_pct >= 40 THEN 'Partially Complete'
                    ELSE 'Blocking'
                END AS predecessor_state
            FROM project_dependencies d
            LEFT JOIN projects pp ON pp.project_id = d.predecessor_project_id
        )
        SELECT
            t.project_id,
            t.department,
            ROUND(t.forecast_delay_weeks, 2) AS forecast_delay_weeks,
            ROUND(t.delay_pct, 2) AS delay_pct,
            COALESCE(GROUP_CONCAT(ps.predecessor_project_id), 'No Predecessor') AS predecessor_ids,
            COALESCE(GROUP_CONCAT(ps.predecessor_state), 'No Predecessor') AS predecessor_statuses
        FROM top5 t
        LEFT JOIN predecessor_status ps
            ON ps.child_project_id = t.project_id
        GROUP BY
            t.project_id, t.department, t.forecast_delay_weeks, t.delay_pct
        ORDER BY
            t.forecast_delay_weeks DESC, t.delay_pct DESC, t.project_id;
        """

        # -----------------------------
        # Query C:
        # Week-over-week growth in total rework hours
        # for entire portfolio.
        # -----------------------------
        query_c = """
        WITH rework_series AS (
            SELECT
                week,
                total_rework_hours,
                LAG(total_rework_hours) OVER (ORDER BY week) AS prev_week_rework
            FROM weekly_rework
        )
        SELECT
            week,
            ROUND(total_rework_hours, 2) AS total_rework_hours,
            ROUND(total_rework_hours - COALESCE(prev_week_rework, 0), 2) AS wow_abs_growth,
            ROUND(
                CASE
                    WHEN prev_week_rework IS NULL OR prev_week_rework = 0 THEN NULL
                    ELSE ((total_rework_hours - prev_week_rework) / prev_week_rework) * 100.0
                END,
                2
            ) AS wow_pct_growth
        FROM rework_series
        ORDER BY week;
        """

        # 3) Execute queries
        result_a = pd.read_sql_query(query_a, conn)
        result_b = pd.read_sql_query(query_b, conn)
        result_c = pd.read_sql_query(query_c, conn)

    print(f"SQLite DB created: {DB_PATH}")
    print("\n=== Query A: Highest Departmental Risk Exposure + Avg Utilization ===")
    print(result_a.to_string(index=False))

    print("\n=== Query B: Top 5 At-Risk Projects + Predecessor Status ===")
    print(result_b.to_string(index=False))

    print("\n=== Query C: Week-over-Week Portfolio Rework Growth ===")
    print(result_c.to_string(index=False))


if __name__ == "__main__":
    main()

SQLite DB created: /workspaces/codespaces-jupyter/portfolio.db

=== Query A: Highest Departmental Risk Exposure + Avg Utilization ===
department  total_risk_exposure  avg_employee_utilization
  Software             66075.89                     99.97

=== Query B: Top 5 At-Risk Projects + Predecessor Status ===
 project_id  department  forecast_delay_weeks  delay_pct predecessor_ids                  predecessor_statuses
         65    Software                455.87      100.0  No Predecessor                        No Predecessor
          9    Hardware                405.78      100.0             3,8 Partially Complete,Partially Complete
         92 Integration                333.55      100.0  No Predecessor                        No Predecessor
         66    Hardware                329.19      100.0  No Predecessor                        No Predecessor
         56    Hardware                327.11      100.0  No Predecessor                        No Predecessor

=== Query C: Week-ove